# 2025년 07월 23일 수요일 (63일차)  


# 📜 목차  
- 개와 고양이 분류 2  
- 개와 고양이 사전학습 1  


<br><br><br>


---

## 🟢 개와 고양이 분류 2  


<br><br>

### 🟡 테이터셋 분리하기  

In [ ]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import tensorflow as tf
import numpy as np
import random
import PIL.Image as pilimg
import imghdr
import pandas as pd
import pickle
import keras
import os
import shutil  # 디렉토리 만들거나 폴더 삭제 등을 담당하는 라이브러리


# 원본데이터셋이 있는 위치 경로
original_dataset_dir = "./data/cats_and_dogs/train"

# 옮길 위치 - 기본 폴더
base_dir = "./data/cats_and_dogs_small"

# 폴더 경로 설정
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
validation_dir = os.path.join(base_dir, "validation")

# ImageDataGenerator나 DataSet이나 둘 다 폴더보고 다동으로 라벨링을 한다.
train_cats_dir = os.path.join(train_dir, "cats")
train_dogs_dir = os.path.join(train_dir, "dogs")

test_cats_dir = os.path.join(test_dir, "cats")
test_dogs_dir = os.path.join(test_dir, "dogs")

validation_cats_dir = os.path.join(validation_dir, "cats")
validation_dogs_dir = os.path.join(validation_dir, "dogs")


# (7월 23일 새롭게 추가한 부분)
# 학습모델(네트워크) - 학습을 완료한 모델을 저장시켜놓고 불러다가 예측을 할 수 있다
model_save_path_keras = "cat_and_dogs_model.keras"  # 확장자가 h5=>keras로 바뀜
# 케라스가 지원한다
history_filepath = "cats_and_dogs_history.bin"
# 학습을 할때마다 정확도, 손실값이 있는 있는 저장해서 던져준다
# 이 값 자체는 저장을 지원하지 않는다. 그래서 pickle을 써서 저장하는데
# 희한하게도 history 자체로 저장하면 에러나고 history.history를 저장해야 에러가 안난다


def ImageCopy():
    # 디텍토리 내의 파일 개수 알아내기
    totalCount = len(os.listdir(original_dataset_dir))
    print(f"전체개수 : {totalCount}")

    # 반복적인 싱행을 위해서 디렉토리 삭제
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir, ignore_errors=True, onerror=None)

    # 디렉토리 생성
    os.makedirs(train_dir)
    os.makedirs(test_dir)
    os.makedirs(validation_dir)

    os.makedirs(train_cats_dir)
    os.makedirs(train_dogs_dir)

    os.makedirs(test_cats_dir)
    os.makedirs(test_dogs_dir)

    os.makedirs(validation_cats_dir)
    os.makedirs(validation_dogs_dir)

    # 디렉토리 내의 파일 개수 알아내기
    trainCount = len(os.listdir(train_dir))
    print(f"훈련셋 개수 : {trainCount}")

    validationCount = len(os.listdir(validation_dir))
    print(f"검증셋 개수 : {validationCount}")

    # 파일 옮기기
    # 옮길 파일명이 cat0.jpg, cat1.jpg, cat2.jpg, ... 이런 식으로 되어 있음
    fnames = [f"cat.{i}.jpg" for i in range(1000)]
    for fname in fnames:
        src = os.path.join(original_dataset_dir, fname)
        dst = os.path.join(train_cats_dir, fname)
        shutil.copyfile(src, dst)  # 1개씩 복사
        print(f"파일 복사 완료 to train : {fname}")

    fnames = [f"cat.{i}.jpg" for i in range(1000, 1500)]
    for fname in fnames:
        src = os.path.join(original_dataset_dir, fname)
        dst = os.path.join(test_cats_dir, fname)
        shutil.copyfile(src, dst)  # 1개씩 복사
        print(f"파일 복사 완료 to test : {fname}")

    fnames = [f"cat.{i}.jpg" for i in range(1500, 2000)]
    for fname in fnames:
        src = os.path.join(original_dataset_dir, fname)
        dst = os.path.join(validation_cats_dir, fname)
        shutil.copyfile(src, dst)  # 1개씩 복사
        print(f"파일 복사 완료 to validation : {fname}")

    fnames = [f"dog.{i}.jpg" for i in range(1000)]
    for fname in fnames:
        src = os.path.join(original_dataset_dir, fname)
        dst = os.path.join(train_dogs_dir, fname)
        shutil.copyfile(src, dst)  # 1개씩 복사

    fnames = [f"dog.{i}.jpg" for i in range(1000, 1500)]
    for fname in fnames:
        src = os.path.join(original_dataset_dir, fname)
        dst = os.path.join(test_dogs_dir, fname)
        shutil.copyfile(src, dst)  # 1개씩 복사

    fnames = [f"dog.{i}.jpg" for i in range(1500, 2000)]
    for fname in fnames:
        src = os.path.join(original_dataset_dir, fname)
        dst = os.path.join(validation_dogs_dir, fname)
        shutil.copyfile(src, dst)  # 1개씩 복사


# ImageCopy()

<br><br>

### 🟡 테이터셋 사용하기 (과대적합 문제 발생)  

In [ ]:
from keras import models, layers


# DataSet 사용하기
def deeplearning():
    model = models.Sequential()
    # 이미지 스케일링
    model.add(layers.Rescaling(1.0 / 255))
    model.add(layers.Conv2D(32, (3, 3), activation="relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation="relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dropout(0.5))  # 중간에 데이터를 절반쯤 없애서 과대적합을 방지
    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    # 데이터셋 - 폴더로부터 이미지 파일을 읽어온다.
    train_ds = keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,  # 훈련셋을 훈련셋과 검증셋으로 8:2로 나눠서 검증
        seed=123,
        subset="training",
        image_size=(180, 180),
        batch_size=16,
    )

    val_ds = keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        seed=123,
        subset="validation",
        image_size=(180, 180),
        batch_size=16,
    )

    # 모델 학습
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
    )

    # 모델 저장
    model.save("./data/cats_dogs.keras")


# deeplearning()

<br><br>

### 🟡 테이터셋 사용하기 (과대적합 문제 해결)  

In [ ]:
from keras import models, layers


# DataSet 사용하기
def deeplearning2():
    # 데이터 증강 파라미터
    data_augmentation = keras.Sequential(
        [
            layers.RandomFlip("horizontal", input_shape=(180, 180, 3)),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
        ]
    )

    model = models.Sequential()
    # 이미지 스케일링
    model.add(layers.Rescaling(1.0 / 255))
    # 과대적합 문제 해결을 위한 추가
    model.add(data_augmentation)
    model.add(layers.Conv2D(32, (3, 3), activation="relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation="relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dropout(0.5))  # 중간에 데이터를 절반쯤 없애서 과대적합을 방지
    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    # 데이터셋 - 폴더로부터 이미지 파일을 읽어온다.
    train_ds = keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,  # 훈련셋을 훈련셋과 검증셋으로 8:2로 나눠서 검증
        seed=123,
        subset="training",
        image_size=(180, 180),
        batch_size=16,
    )

    val_ds = keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        seed=123,
        subset="validation",
        image_size=(180, 180),
        batch_size=16,
    )

    # 모델 학습
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
    )

    # 모델 저장
    model.save("./data/cats_dogs2.keras")


# deeplearning2()

### 🟡 모델 저장 및 히스토리 구현하기  

In [ ]:
from keras import models, layers


# DataSet 사용하기
def deeplearning3():
    # 데이터 증강 파라미터
    data_augmentation = keras.Sequential(
        [
            layers.RandomFlip("horizontal", input_shape=(180, 180, 3)),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
        ]
    )

    model = models.Sequential()
    # 이미지 스케일링
    model.add(layers.Rescaling(1.0 / 255))
    # 과대적합 문제 해결을 위한 추가
    model.add(data_augmentation)
    model.add(layers.Conv2D(32, (3, 3), activation="relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation="relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dropout(0.5))  # 중간에 데이터를 절반쯤 없애서 과대적합을 방지
    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    # 데이터셋 - 폴더로부터 이미지 파일을 읽어온다.
    train_ds = keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,  # 훈련셋을 훈련셋과 검증셋으로 8:2로 나눠서 검증
        seed=123,
        subset="training",
        image_size=(180, 180),
        batch_size=16,
    )

    val_ds = keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        seed=123,
        subset="validation",
        image_size=(180, 180),
        batch_size=16,
    )

    # 모델 학습
    history = model.fit(train_ds, validation_data=val_ds, epochs=1)  # 임시 epochs=1

    # 모델 저장
    try:
        model.save(model_save_path_keras)  # 모델 저장하기
        print("모델 저장 성공")
    except Exception as e:
        print(f"모델 저장 중 오류 발생: {e}")

    # 히스토리 저장하기
    try:
        with open(history_filepath, "wb") as f:
            pickle.dump(history.history, f)
        print("히스토리 저장 성공")
    except Exception as e:
        print(f"히스토리 저장 중 오류 발생: {e}")


deeplearning3()

### 🟡 차트 그리기  

In [ ]:
import matplotlib.pyplot as plt


def draw_chart():
    print("저장된 모듈 불러오기")
    try:
        loaded_model_keras = keras.models.load_model(model_save_path_keras)
        print("모델 부르기 성공")
    except Exception as e:
        print(f"모델 로딩중 실패 : {e}")

    print("히스토리 불러오기")
    try:
        with open(history_filepath, "rb") as file:
            history = pickle.load(file)
            print("히스토리 로딩 성공")
    except Exception as e:
        print(f"히스토리 로딩중 실패 : {e}")

    # 히스토리의 키값들 가져오기 - 에포크회수만큼 list로 가져온다
    acc = history["accuracy"]
    val_acc = history["val_accuracy"]
    loss = history["loss"]
    val_loss = history["val_loss"]

    # x축 좌표값 만들기
    X = range(len(acc))

    plt.plot(X, acc, "ro", label="Training accuracy")
    plt.plot(X, val_acc, "bo", label="Validation accuracy")
    plt.title("Training and validation accuracy")

    # 새로운 창을 열어 차트를 그린다.
    plt.figure(figsize=(10, 5))
    plt.plot(X, loss, "ro", label="Training loss")
    plt.plot(X, val_loss, "bo", label="Validation loss")
    plt.title("Training and validation loss")

    plt.show()  # 전체 한번에 출력


draw_chart()

### 🟡 예측하기  

In [ ]:
def predict():  # 예측하기
    # 1. 학습된 모듈을 불러온다.
    loded_model_keras = None

    try:
        loaded_model_keras = keras.models.load_model(model_save_path_keras)
        print("모델 불러오기 성공")
    except Exception as e:
        print(f"모델 불러오기 실패: {e}")
        return

    # 예측데이터셋 만들기
    val_ds = keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        seed=123,
        subset="validation",
        image_size=(180, 180),
        batch_size=16,
    )

    print("-------- 라벨링 확인하기 --------")
    class_names = val_ds.class_names
    print(class_names)
    print("-------- 라벨링 확인 끝 --------")

    total_match_count = 0  # 전체 일치한 개수
    total_samples_proceed = 0  # 전체 처리한 샘플 개수
    max_smaples_to_process = 500  # 데이터를 500개까지만 예측해보자
    for input_batch, labels_batch in val_ds:
        # val_ds가 폴더로 부터 이미지 파일을 읽어오는데 batch_size만큼 읽어온다.
        total_samples_proceed, len(labels_batch)

        # 예측하기
        predictions_batch = loaded_model_keras.predict(input_batch)
        print(predictions_batch)

        # 예측 결과와 실제 레이블 비교
        # 이진분류라서 결과값이 하나가 나온다. 꽅 분류 같으면 맞느다. [0.3, 0.7] [0.3]
        # 이진분류일 때는 라벨이 1인 요소의 확률을 전달한다.
        # 다중분류일때는 [0.1, 0.1, 0.6, 0.1, 0.1]
        predicted_class = (predictions_batch > 0.5).astype(
            int
        )  # 0.5 보다 큰거는 True, 작은 거는 False
        print(f"예측 결과: {predicted_class}")
        print(f"실제 레이블: {labels_batch.numpy()}")  # Tensor -> numpy로 변환

        match_count = np.sum(predicted_class == labels_batch.numpy())
        total_match_count += match_count

    # print(f"{total_samples_proceed}개의 샘플을 처리했습니다.")
    # print(f"{len(labels_batch)}개의 라벨을 처리했습니다.")
    print(f"전체 데이터 개수: {total_samples_proceed}")
    print(f"맞춘 개수: {total_match_count}")
    print(f"못 맞춘 개수: {total_samples_proceed - total_match_count}")


predict()

### 🟡 선택 함수 만들기  

In [ ]:
def main():
    while True:
        print("1. 파일복사")
        print("2. 학습")
        print("3. 차트")
        print("4. 예측")
        print("5. 종료")

        sel = input("번호를 입력하세요: ")

        if sel == "1":
            ImageCopy()
        elif sel == "2":
            deeplearning3()
        elif sel == "3":
            pass
        elif sel == "4":
            pass
        elif sel == "5":
            break
        else:
            print("잘못된 번호입니다. 다시 입력해주세요.")


if __name__ == "__main__":
    main()